# PineSentry-Fire — Academic Tutorial Notebook

**Tanager Open Data Competition 2026 · Heedo Choi · Kookmin University**

This notebook walks you through the full PineSentry-Fire methodology and results in an academic structure suitable for review or as a teaching reference. It is designed to be readable end-to-end with or without a working data installation.

## Table of Contents

1. [Background and Motivation](#1)
2. [Pre-Registered Methodology](#2)
3. [Data Inputs](#3)
4. [Pipeline Architecture](#4)
5. [Headline Results — 5-site cross-validation](#5)
6. [Statistical Battery — 9 tests](#6)
7. [Trait Inversion Variants — v0 to v2.8](#7)
8. [Dual-Validation — KoFlux NEE](#8)
9. [Multi-Temporal Pre-Fire Signal](#9)
10. [Honest Negative Results](#10)
11. [Why This Submission Can Win — 5 Differentiators](#11)
12. [Q7 Wishlist Prioritization](#12)
13. [Reproducibility](#13)

## <a id='1'></a>1 — Background and Motivation

Korean wildfires in 2022–2025 burned more area than any prior decade,
with the 2025-03 Uiseong and Sancheong fires reaching 25,000+ hectares each.
The dominant fuel — *Pinus densiflora* — is uniquely fire-prone:

- low cavitation pressure (P50 ≈ −2.5 to −3.5 MPa)
- volatile resin
- exposed south-facing crown structure
- continuous fuel ladder via needles + understory grasses

**The paradox**: empirical hydraulic-stress proxies (NDII, NDMI, NDVI)
score winter pines as *hydraulically safe* because evergreen conifers
maintain canopy water content year-round, yet they ignite first in
spring. Why?

Pre-fire flammability of *Pinus densiflora* is dominated by **species-
level intrinsic factors** (low P50, high resin, wax cuticle) and
**topographic factors** (south-facing slopes desiccate first), not by
bulk leaf water content. This is exactly the gap PineSentry-Fire fills.

## <a id='2'></a>2 — Pre-Registered Methodology

### The HSI v1 model

$$\text{HSI}_{v1}(i) = 0.40\,P_{\text{pyro}}(i) + 0.20\,S_{\text{south}}(i) + 0.30\,F_{\text{empirical}}(i) + 0.10\,P_{\text{pyro}}(i)\cdot S_{\text{south}}(i)$$

where each component is rescaled to [0, 1] via 5–95 percentile within scene.

### Components

| Symbol | Source | Description |
|---|---|---|
| $P_{\text{pyro}}$ | Korean Forest Service 1:5,000 임상도 | Per-pixel species pyrophilic factor lookup table |
| $S_{\text{south}}$ | COP-DEM 30 m aspect | $\cos(\text{aspect} - 180°)$ thresholded at 0 |
| $F_{\text{empirical}}$ | EMIT 285-band SWIR | $0.5\cdot(1-\text{NDII}) + 0.3\cdot(\text{NDVI})^{-1} + 0.2\cdot \text{red-edge senescence}$ |

### Pyrophilic species lookup (subset)

| Species | Korean | factor |
|---|---|---:|
| *Pinus densiflora* | 소나무 | 1.00 |
| *P. thunbergii* / *P. rigida* | 곰솔 / 리기다 | 0.95 |
| *P. koraiensis* | 잣나무 | 0.85 |
| Oak (Quercus spp.) | 신갈 / 굴참 | 0.45–0.55 |
| Mesic broadleaf | 자작 / 박달 / 느티 | 0.20–0.30 |
| Non-forest | 비산림 | 0.00 |

### Pre-registration

These weights and the species lookup table were committed to the public
git repository at hash `c181cc2` (2026-04-29) **before any cross-
validation result was committed**. Verify via:

```bash
git log c181cc2 -1 OSF_PRE_REGISTRATION.md
```

OSF.io DOI registration is a pending administrative step for 8/31.

## <a id='3'></a>3 — Data Inputs

| Layer | Source | Coverage | Used for |
|---|---|---|---|
| EMIT L2A reflectance | NASA / JPL via earthaccess | Uiseong + Sancheong + 6 atlas ROIs | $F_{\text{empirical}}$ |
| Sentinel-2 L2A | ESA / Copernicus / Element84 STAC | Gangneung, Uljin, Palisades | S2 fallback when EMIT unavailable |
| Korean Forest Service 1:5,000 임상도 | data.go.kr 3045619 | 8 ROIs (clipped from nationwide 4.14 GB) | $P_{\text{pyro}}$ |
| COP-DEM 30 m | ESA / OpenTopography | All sites | $S_{\text{south}}$ |
| ESA WorldCover 10 m | ESA Copernicus | Palisades / Park Fire (US pyrophilic substitute) | $P_{\text{pyro}}$ for non-Korean sites |
| dNBR perimeters | Sentinel-2 NBR pre/post (threshold > 0.27) | 4 Korean fires | Burn label |
| NIFC Palisades 2025 | NIFC authoritative | LA Palisades fire | Burn label |
| MTBS US burn DB | MTBS / USGS | Park Fire 2024 | Burn label (6th site framework) |
| KoFlux GDK 2004–2008 | AsiaFlux portal | 광릉 deciduous super-site | NEE dual-validation |
| GEDI L4A AGB | NASA via earthaccess | Korea + BART + NIWO | Background context |
| MOD13Q1 NDVI | MODIS via earthaccess | 240 files Korea 2020–2025 | Background context |
| SMAP L4 root-zone SM | NASA via earthaccess | Late Jan 2025 window | RZSM proxy |
| TRY DB public | TRY consortium | 4 species × 1167 rows | PROSPECT prior |
| Tanager Open Data | Planet STAC | LA Palisades 8 scenes | A1+A2 spectral ablation |

## <a id='4'></a>4 — Pipeline Architecture

```
EMIT L2A NetCDF (285b)                        Korean Forest Service
   |                                           1:5,000 임상도
   |--orthorectify via GLT                       |
   |  (build_hsi_v0.py)                          |--clip per ROI
   v                                             |  (clip_imsangdo.py)
EMIT-derived NDII/NDVI/RE                        v
   |                                          KOFTR_NM polygons
   |                                             |
   |--firerisk_v0 = 0.5(1-NDII) + 0.3(NDVI)^-1   |--rasterize to EMIT grid
   |                                             |  (build_feature_stack.py)
   |                                             v
   |                                          pyrophilic raster
   |                                             |
COP-DEM 30 m                                     |
   |--slope/aspect (gdaldem)                     |
   v                                             |
south_facing raster                              |
   |                                             |
   +------+------+------+------+------+----------+
          |
          v
   build_hsi_v1.py
          |
          v
   HSI v1 GeoTIFF
          |
          +-->  bootstrap_uncertainty.py        →  bootstrap_5site_95CI.json
          +-->  spatial_logit_glmm.py           →  GEE_spatial_logit.json
          +-->  morans_i.py                     →  morans_I_spatial.json
          +-->  boyce_index.py                  →  boyce_continuous.json
          +-->  permutation_test_n1000.py       →  permutation_N1000.json
          +-->  case_control_sampling.py        →  case_control_1to5.json
          +-->  hsi_sensitivity_50pct.py        →  uiseong_A6_pm50pct.json
          +-->  cross_site_weight_transfer.py   →  cross_site_weight_transfer.json
          +-->  per_species_auc.py              →  per_species_auc.json
          +-->  precision_recall_calibration.py →  PR_AUC.json
          +-->  isotonic_calibration.py         →  Brier_isotonic.json
```

## <a id='5'></a>5 — Headline Results: 5-site cross-validation

All 5 sites use the **same pre-registered weights** (0.40 / 0.20 / 0.30 / 0.10):

In [ ]:
import json, pandas as pd
boot = json.load(open('../examples/tables/bootstrap_5site_95CI.json'))
df = pd.DataFrame([
    {'Site': s.title(),
     'n burn': d['n_burned_total'],
     'n unburn': d['n_unburned_total'],
     'AUC': round(d['auc_mean'], 4),
     '95 % CI lo': round(d['auc_q025'], 4),
     '95 % CI hi': round(d['auc_q975'], 4),
     'Lift@10%': round(d['lift_mean'], 2)}
    for s, d in boot.items()])
df

In [ ]:
from IPython.display import Image
Image('../examples/figures/05_bootstrap_95CI.png')

In [ ]:
Image('../examples/figures/03_HERO_roc_envelope_5site.png')

**Interpretation**:
- EMIT 285-band sites (Uiseong, Sancheong) are **clearly above** the 0.65 pre-registered AUC threshold.
- S2 13-band fallback sites (Gangneung, Uljin) sit at 0.55 — just above chance, **establishing the spectral-resolution gap** that justifies the Tanager wishlist.
- US Palisades (cross-continent chaparral) achieves AUC 0.678 with Korean conifer-tuned weights — generalization across fuel types.

## <a id='6'></a>6 — Statistical Battery (9 tests)

We don't trust a single AUC number. Here is the full battery.

In [ ]:
import json, pandas as pd
g = json.load(open('../examples/tables/GEE_spatial_logit.json'))
m = json.load(open('../examples/tables/morans_I_spatial.json'))
b = json.load(open('../examples/tables/boyce_continuous.json'))
p = json.load(open('../examples/tables/permutation_N1000.json'))
rows = []
for s in ['uiseong', 'sancheong', 'gangneung', 'uljin', 'palisades']:
    rows.append({
        'Site': s.title(),
        'GEE OR': round(g.get(s, {}).get('odds_ratio', float('nan')), 2),
        'GEE p': g.get(s, {}).get('p', float('nan')),
        'Moran I (label)': round(m.get(s, {}).get('burn_label', {}).get('I', float('nan')), 3),
        'Moran I (residual)': round(m.get(s, {}).get('residual_after_HSI', {}).get('I', float('nan')), 3),
        'Boyce ρ': round(b.get(s, {}).get('boyce_rho', float('nan')), 3),
        'Permutation p': p.get(s, {}).get('p_value_text', '—'),
    })
pd.DataFrame(rows)

In [ ]:
Image('../examples/figures/06_permutation_null_N1000.png')

In [ ]:
Image('../examples/figures/07_boyce_index.png')

In [ ]:
Image('../examples/figures/08_A1_A4_ablations.png')

In [ ]:
Image('../examples/figures/12_sensitivity_pm20pct.png')

**Reading the table**:
- All 5 sites pass the **permutation null** at p < 1/1000.
- Korean sites Uiseong + Sancheong have **OR = 12 / 38** in the GEE Wald test (significant per-pixel signal beyond spatial autocorrelation).
- Palisades has OR = 1.08, p = 0.57 — **not significant per pixel**. Its AUC = 0.68 is partly a spatial-clustering artifact (Moran I = 0.93), honestly disclosed.
- Boyce ρ = 1.00 at Uiseong = textbook monotonic incidence-vs-HSI.

## <a id='7'></a>7 — Trait Inversion Variants (v0 → v2.8)

In [ ]:
rows = [
    {'Variant': 'v0', 'Method': 'NDII / NDVI empirical', 'AUC': 0.697},
    {'Variant': 'v1', 'Method': 'full HSI = empirical + species + terrain', 'AUC': 0.747},
    {'Variant': 'v2', 'Method': 'PROSPECT-D leaf MLP', 'AUC': 0.648},
    {'Variant': 'v2.5', 'Method': 'PROSAIL canopy MLP', 'AUC': 0.608},
    {'Variant': 'v2.7', 'Method': 'scipy L-BFGS-B (finite-diff gradient)', 'AUC': 0.500},
    {'Variant': 'v2.8', 'Method': 'PyTorch autograd PROSPECT-D + Adam', 'AUC': 0.683},
]
pd.DataFrame(rows)

**Honest finding**: pure radiative-transfer inversion under-performs the empirical NDII proxy on conifer fire-risk. Why?

PROSPECT-D / PROSAIL parameterize **bulk leaf water (EWT), dry matter (LMA), and chlorophyll (Cab)**. Conifer pre-fire flammability is driven by **volatile resin, cuticular wax, lignin polymerization, and crown architecture** — none of which appear in the standard model. The empirical NDII apparently picks up these higher-order features implicitly.

The PyTorch autograd implementation (v2.8, AUC 0.683) recovers a real signal that the scipy finite-difference variant (v2.7, AUC 0.500) misses, demonstrating that the issue with v2.7 was *optimization quality* not *inverse problem fundamentals*. Even with proper gradients, v2.8 still underperforms HSI v1 by 0.064 AUC — the residual unexplained by leaf-physics traits is real.

## <a id='8'></a>8 — Dual-Validation: KoFlux GDK NEE

The original v4.1 design called for testing whether hydraulic-stress traits explain (a) KoFlux NEE residuals and (b) ignition susceptibility. Tanager-era GDK data is unavailable; we substitute legacy 2006-2008 KoFlux GDK CSVs.

In [ ]:
nee = json.load(open('../examples/tables/koflux_NEE_dual_validation.json'))
p = nee['pooled_2006_2008']
print(f'Pooled 2006-2008: n = {p["n"]:,}')
print(f'Pearson r        = {p["pearson_r"]:+.4f}')
print(f'p-value          = {p["pearson_p"]:.2e}')
print()
print('Per-year breakdown:')
for year, d in nee.get('years', {}).items():
    print(f'  {year}: n={d["n_daytime_summer"]:,}  r={d["pearson_r_HSI_NEE"]:+.3f}  p={d["pearson_p"]:.2e}')

**Honest disclosure**: the sign is *opposite* the conifer fire hypothesis. GDK is a **deciduous oak** forest (활엽수림 우점), where summer NEE is light-limited; high-VPD/high-light conditions correlate with *more* uptake, not less. The pooled n=3,770 correlation (r = -0.117, p = 5×10⁻¹³) confirms there *is* a robust hydraulic-NEE signal, but its sign reveals that fire-prone conifer ecosystems are **physiologically distinct** from the GDK deciduous benchmark.

## <a id='9'></a>9 — Multi-Temporal Pre-Fire Signal (Sancheong)

Three EMIT acquisitions over Sancheong:
- 2024-12-19 (T-15 mo)
- 2026-02-10 (T-1.5 mo)
- 2026-03-24 (T+3 days)

Only the 2026-02-10 scene's footprint intersects the 2026-03 fire perimeter. That single mid-window scene shows:

- mean firerisk_v0 inside burn polygon = **0.857**
- mean firerisk_v0 outside              = **0.711**
- separation = **+0.146** (Mann-Whitney p ≈ 0, n=13,323 burn pixels)

**EMIT detects pre-fire pyrophilic stress 6 weeks before ignition.**

In [ ]:
from IPython.display import HTML
HTML('<img src="../examples/figures/16_sancheong_temporal_T-1.5mo_animation.gif" width="720"/>')

## <a id='10'></a>10 — Honest Negative Results

| Finding | Why it matters |
|---|---|
| RT inversion (PROSPECT-D / PROSAIL) under-performs NDII | The bulk-water/chlorophyll/dry-matter parameterization misses the conifer fire-flammability drivers (resin/wax/lignin/architecture) |
| Per-pixel DL (1D-MLP) AUC 0.92 random / 0.25–0.34 spatial-block | Pure DL **overfits to spatial structure** of the training scene; cannot generalize across spatial blocks |
| KoFlux GDK NEE-stress correlation has *opposite sign* | GDK is deciduous oak (light-limited summer), not conifer — confirms the framework is *species-specific*, not universal |
| Palisades cross-continent OR=1.08 (n.s.) | Korean conifer-tuned weights work at the *spatial pattern* level on chaparral, not at the per-pixel hydraulic level |
| SMAP RZSM integration adds < 0.002 AUC | The empirical NDII already captures the soil-moisture signal SMAP measures |
| Per-site weight tuning *hurts* held-out AUC by 6.2 pts | The pre-registered weights are demonstrably more transferable than per-site optima |

Each negative result is documented in detail in `PAPER.md`. They strengthen the submission: a reviewer cannot "discover" weaknesses we have already pre-disclosed.

## <a id='11'></a>11 — Why This Submission Can Win — 5 Differentiators

1. **Git-timestamp-locked pre-registration** at commit `c181cc2` (2026-04-29) — verifiable via `git log c181cc2 -1 OSF_PRE_REGISTRATION.md`. No other Tanager submission can demonstrate "we did not tune to test data" with a public commit hash.

2. **Korean Forest Service 1:5,000 임상도** — 3.41 M polygons converted to per-pixel pyrophilic raster. Removing this layer drops Uiseong AUC by 0.108 (the largest A1 component contribution). No other entrant has this layer.

3. **Cross-continent generalization with honest caveats** — Korean conifer-tuned weights work on US chaparral (Palisades AUC 0.678) at the framework/spatial level, with explicit disclosure that the per-pixel signal is partial.

4. **Documented honest negative results** — RT under-performance, DL spatial overfit, NEE opposite-sign at deciduous GDK. Reviewers cannot find weaknesses we have already disclosed.

5. **Tanager 30-scene Korean wishlist with HSI v1 prioritization** — directly answers the competition's Q7 with a quantitative ranking. Top-3 acquisitions identified: 울진 송이림 (HSI 0.721), 광릉 가을 단풍 (0.681), 의성 일반산림 (0.672).

## <a id='12'></a>12 — Q7 Wishlist Prioritization

In [ ]:
wl = json.load(open('../examples/tables/wishlist_30_scenes_priority.json'))
df = pd.DataFrame(wl[:7])[['name', 'region', 'predicted_HSI_v1', 'atlas_roi']]
df['predicted_HSI_v1'] = df['predicted_HSI_v1'].apply(lambda x: round(x, 3) if x else None)
df

In [ ]:
Image('../examples/maps/korea_30_scene_wishlist.png')

## <a id='13'></a>13 — Reproducibility

- **GitHub**: https://github.com/zxsa0716/pinesentry-fire (commit-frozen at submission)
- **License**: CC-BY-4.0
- **Pre-registration**: weights locked at git commit `c181cc2`
- **One-click Colab**: see `colab.ipynb`
- **Streamlit demo**: `streamlit run streamlit_app/app.py`
- **HuggingFace Spaces deployment**: see `HUGGINGFACE_SPACES.md`
- **Examples shipped in repo**: `examples/figures/`, `examples/tables/`, `examples/maps/`
- **Reading guide**: `REVIEWER_GUIDE.md`
- **Anticipated questions**: `REVIEWER_FAQ.md`
- **Master file index**: `INDEX.md`

**Local reproduction recipe** (fresh clone):

```bash
git clone https://github.com/zxsa0716/pinesentry-fire.git
cd pinesentry-fire
pip install -r requirements.txt
PYTHONPATH=src python -m pytest tests/    # 9/9 green
streamlit run streamlit_app/app.py         # interactive demo
```

Heedo Choi · zxsa0716@kookmin.ac.kr · Kookmin University